In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/GPT-from-Scratch/notebooks/"
os.chdir(PROJECT_DIR)
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/GPT-from-Scratch/notebooks


In [2]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from functools import partial

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
tokenizer = Tokenizer.from_file("../src/tokenizer/arabic_bpe_tokenizer.json")

with open("../data/finetune/sft_data.json", "r", encoding="utf-8") as f:
    sft_data = json.load(f)

print("Number of SFT samples:", len(sft_data))
print(sft_data[0])

Number of SFT samples: 200
{'instruction': 'أعد صياغة الجملة التالية بلغة أبسط.', 'input': 'كان الطفل سعيدًا لأنه وجد كتابًا جديدًا.', 'output': 'فرح الطفل لأنه عثر على كتاب جديد.'}


In [5]:
def format_input(entry):
    instruction_text = (
        "فيما يلي تعليمات تصف مهمة. "
        "اكتب استجابة مناسبة تكمل الطلب."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [6]:
example = sft_data[0]
formatted = format_input(example)
response = f"\n\n### Response:\n{example['output']}"

print(formatted + response)

فيما يلي تعليمات تصف مهمة. اكتب استجابة مناسبة تكمل الطلب.

### Instruction:
أعد صياغة الجملة التالية بلغة أبسط.

### Input:
كان الطفل سعيدًا لأنه وجد كتابًا جديدًا.

### Response:
فرح الطفل لأنه عثر على كتاب جديد.


In [7]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []

        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text

            encoded = tokenizer.encode(full_text).ids
            self.encoded_texts.append(encoded)

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [8]:
train_portion = int(len(sft_data) * 0.85)
test_portion = int(len(sft_data) * 0.10)
val_portion = len(sft_data) - train_portion - test_portion

train_data = sft_data[:train_portion]
test_data = sft_data[train_portion:train_portion + test_portion]
val_data = sft_data[train_portion + test_portion:]

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

Train: 170
Val: 10
Test: 20


In [9]:
def custom_collate_fn(
    batch,
    pad_token_id=0,        # we will use tokenizer PAD token id next
    ignore_index=-100,
    allowed_max_length=128,
    device="cpu"
):
    batch_max_length = min(max(len(item) + 1 for item in batch), allowed_max_length + 1)

    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        padded = padded[:batch_max_length]

        inputs = torch.tensor(padded[:-1], dtype=torch.long)
        targets = torch.tensor(padded[1:], dtype=torch.long)

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()

        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [10]:
vocab = tokenizer.get_vocab()

PAD_ID = vocab["[PAD]"]
print("PAD_ID:", PAD_ID)

PAD_ID: 0


In [11]:
customized_collate_fn = partial(
    custom_collate_fn,
    pad_token_id=PAD_ID,
    allowed_max_length=128,
    device=device
)

train_dataset = InstructionDataset(train_data, tokenizer)
val_dataset = InstructionDataset(val_data, tokenizer)
test_dataset = InstructionDataset(test_data, tokenizer)

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 21
Val batches: 2
Test batches: 3


In [12]:
for inputs, targets in train_loader:
    print("Inputs shape:", inputs.shape)
    print("Targets shape:", targets.shape)
    print("\nFirst input sample:\n", inputs[0][:40])
    print("\nFirst target sample:\n", targets[0][:40])
    break

Inputs shape: torch.Size([8, 95])
Targets shape: torch.Size([8, 95])

First input sample:
 tensor([ 730, 2660,  105,  292,  778, 3641,  127, 3984,   11,  102, 1162,  406,
         107,  774, 2894,  912,  227,  521,  625,   11,    1,    1,    1, 1459,
          74,   75,   73,   76,   58, 1874,   23,  102, 1162,  748,  234, 3838,
         170,  598, 4346,  210], device='cuda:0')

First target sample:
 tensor([2660,  105,  292,  778, 3641,  127, 3984,   11,  102, 1162,  406,  107,
         774, 2894,  912,  227,  521,  625,   11,    1,    1,    1, 1459,   74,
          75,   73,   76,   58, 1874,   23,  102, 1162,  748,  234, 3838,  170,
         598, 4346,  210,  229], device='cuda:0')
